# Báo cáo tài chính chuẩn hoá FireAnt (financial-data)

Bộ chỉ tiêu "3 trong 1" từ `restv2.fireant.vn/symbols/{ticker}/financial-data`: CĐKT + KQKD +
LCTT đã gộp và chuẩn hoá sẵn thành **một dòng một kỳ**, kèm chỉ số định giá (P/E, P/B, EV/EBITDA),
sinh lời (ROE, ROA, ROIC), tăng trưởng, số liệu trung bình ngành, Piotroski F-Score và
Altman Z-Score — khác với `full-financial-reports` (bảng dài, chỉ tiêu thô theo cây phân cấp)
mà `Fireant.get_financials` đang dùng.

Lưu vào `fscore.db`, **giữ nguyên tên trường của API làm tên cột**. Bộ trường phụ thuộc loại hình
doanh nghiệp nên mỗi loại một bảng, khoá `(symbol, period)`:

| bảng | loại hình | số trường |
|---|---|---|
| `fireant_financial_data_general` | General | 340 |
| `fireant_financial_data_bank` | Bank | 298 |
| `fireant_financial_data_securities` | Securities | 449 |
| `fireant_financial_data_insurance` | Insurance | 419 |

Vì sao tách bảng: giữa các loại hình có tên trường chỉ khác nhau ở chữ hoa/thường
(`ShortTermPrepaidExpense` vs `ShorttermPrepaidExpense`), mà SQLite **không phân biệt hoa thường**
ở tên cột nên không nhét chung một bảng được.

Cột `period` dùng chung nhãn kỳ với các bảng `fireant_*` khác: `'2025'` (năm) hoặc `'Q1-2026'`
(quý), nên dữ liệu năm và quý nằm chung bảng mà không đụng khoá.

In [1]:
from fireant_connector import Fireant
import pandas as pd
import sqlite3

# financial-data hỗ trợ cả ngân hàng / chứng khoán / bảo hiểm (mỗi loại một bảng)
# nên crawl toàn bộ danh sách mã, không lọc phi tài chính như notebook fireant.ipynb
tickers = pd.read_csv('./data/tickers_all.csv')
tickers

,ID,Symbol,CompanyName,Exchange,IndustryName1,IndustryName2,IndustryName3,FirstTradeDate,BeforeExchange,DelistedReason,IsTerminated
0,4712.0,A32,CTCP 32,UPCOM,Tiêu dùng không thiết yếu,Thời trang và hàng lâu bền,Thời trang và dệt may,2018-10-22 17:00:00,NaN,NaN,False
1,2541.0,AAA,CTCP Nhựa An Phát Xanh,HOSE,Nguyên vật liệu,Nguyên vật liệu,Hóa chất,2016-11-24 17:00:00,HNX,"AAA: Ngày 18/11/2016, ngày hủy niêm yết cổ phi...",False
2,5601.0,AAH,CTCP Hợp Nhất,UPCOM,Năng lượng,Năng lượng,Dầu khí,2024-01-10 17:00:00,NaN,NaN,False
3,1813.0,AAM,CTCP Thủy sản MeKong,HOSE,Tiêu dùng thiết yếu,"Thực phẩm, đồ uống và thuốc lá",Thực phẩm,2009-09-23 17:00:00,NaN,NaN,False
4,5816.0,AAN,CTCP Lương thực A An,HOSE,Tiêu dùng thiết yếu,"Thực phẩm, đồ uống và thuốc lá",Thực phẩm,2026-05-21 17:00:00,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
2044,NaN,VFW,Công ty Cổ phần Quản lý quỹ Lộc Việt,OTC,NaN,NaN,NaN,NaN,NaN,NaN,True
2045,NaN,VHW,CTCP Nước Khoáng Vĩnh Hảo,OTC,NaN,Hàng tiêu dùng,NaN,NaN,NaN,NaN,True
2046,NaN,VLD,Công ty Cổ phần Bất Động Sản Viettronics,OTC,NaN,NaN,NaN,NaN,NaN,NaN,True
2047,NaN,VTT,NaN,HNX,NaN,NaN,NaN,NaN,NaN,NaN,True


In [2]:
symbols = tickers['Symbol'].dropna().unique().tolist()
len(symbols)

2049

## Thử một mã trước

API **bắt buộc** tham số `count` và **không có tham số neo năm** — luôn trả `count` kỳ gần nhất
rồi tự cắt khi hết dữ liệu (HPG xin 100 kỳ chỉ ra 22 kỳ, 2025 → 2004). `get_financial_data` tự
tính `count` từ **năm hiện tại** (không phải `end_year`) rồi lọc lại theo `start_year`.

In [3]:
fa = Fireant(request_sleep=0.2)

hpg = fa.get_financial_data('HPG', start_year=2009)
print(hpg.shape, '->', len(hpg), 'kỳ x', hpg.shape[1] - 2, 'trường + symbol/period')
hpg[['symbol', 'period', 'CompanyType', 'NetSale', 'ProfitAfterTax',
     'TotalAsset', 'ROE', 'PE', 'PB', 'PiotroskiFScore', 'ManufacturingZScore']]

HPG financial-data NAM: xin 19 ky (>= 2009)


(17, 342) -> 17 kỳ x 340 trường + symbol/period


,symbol,period,CompanyType,NetSale,ProfitAfterTax,TotalAsset,ROE,PE,PB,PiotroskiFScore,ManufacturingZScore
0,HPG,2009,General,8123394614746,1270706623417,10243239989085,0.27149,9.03169,2.38074,4,2.88218
1,HPG,2010,General,14267083816361,1376316086778,14903658232099,0.22864,9.11633,1.92257,3,2.50832
2,HPG,2011,General,17851896561575,1296850503678,17524683026075,0.1682,4.43882,0.74029,4,2.10407
3,HPG,2012,General,16826851892984,1030505429517,19015763461546,0.12019,8.85301,1.08843,4,1.97564
4,HPG,2013,General,18934292150531,2010435402769,23076377862689,0.21517,8.81341,1.81289,8,2.2071
5,HPG,2014,General,25525348822713,3250214590204,22089104397803,0.29178,8.12312,2.16524,7,3.69145
6,HPG,2015,General,27452932114333,3504382487779,25506769185545,0.26373,6.13982,1.48161,7,3.18201
7,HPG,2016,General,33283210159987,6606202726929,33226552317885,0.38477,5.50815,1.84179,7,4.04817
8,HPG,2017,General,46161691614304,8014756586048,53022184778251,0.30649,8.876,2.20111,4,4.22813
9,HPG,2018,General,55836458379759,8600550706227,78223007670925,0.23481,7.66766,1.62325,2,2.52143


## Crawl toàn bộ

`skip_existing=True` bỏ qua mã đã có trong DB nên chạy lại được sau khi đứt giữa chừng; mọi lỗi
của một mã chỉ được ghi vào log chứ không làm dừng cả đợt, và Ctrl+C vẫn trả về phần đã chạy.

In [4]:
log = fa.crawl_financial_data_to_db(
    symbols,
    start_year=2009,
    type_time='NAM',
    db_path='fscore.db',
    skip_existing=True,
    log_path='./logs/fireant_financial_data.csv',
    sleep=0.3,
)
log

A32 financial-data NAM: xin 19 ky (>= 2009)
Da luu 10 ky (A32) vao cafef.db.fireant_financial_data_general
(1/2049) A32 xong (10 ky)
AAA financial-data NAM: xin 19 ky (>= 2009)
Da luu 17 ky (AAA) vao cafef.db.fireant_financial_data_general
(2/2049) AAA xong (17 ky)
AAH financial-data NAM: xin 19 ky (>= 2009)
Da luu 7 ky (AAH) vao cafef.db.fireant_financial_data_general
(3/2049) AAH xong (7 ky)
AAM financial-data NAM: xin 19 ky (>= 2009)
Da luu 17 ky (AAM) vao cafef.db.fireant_financial_data_general
(4/2049) AAM xong (17 ky)
AAN financial-data NAM: xin 19 ky (>= 2009)
Da luu 2 ky (AAN) vao cafef.db.fireant_financial_data_general
(5/2049) AAN xong (2 ky)
AAS financial-data NAM: xin 19 ky (>= 2009)
Da luu 10 ky (AAS) vao cafef.db.fireant_financial_data_securities
(6/2049) AAS xong (10 ky)
AAT financial-data NAM: xin 19 ky (>= 2009)
Da luu 8 ky (AAT) vao cafef.db.fireant_financial_data_general
(7/2049) AAT xong (8 ky)
AAV financial-data NAM: xin 19 ky (>= 2009)
Da luu 12 ky (AAV) vao cafef

,Symbol,IsSuccess,Periods,CompanyType,ErrorMessage
0,A32,True,10,General,NaN
1,AAA,True,17,General,NaN
2,AAH,True,7,General,NaN
3,AAM,True,17,General,NaN
4,AAN,True,2,General,NaN
...,...,...,...,...,...
2044,VFW,True,0,NaN,Khong co du lieu
2045,VHW,True,0,NaN,Khong co du lieu
2046,VLD,True,0,NaN,Khong co du lieu
2047,VTT,True,0,NaN,Khong co du lieu


In [5]:
# mã lỗi / mã không có dữ liệu
log[~log['IsSuccess'] | (log['Periods'] == 0)]

,Symbol,IsSuccess,Periods,CompanyType,ErrorMessage
398,E1VFVN30,True,0,NaN,Khong co du lieu
443,FUCTVGF3,True,0,NaN,Khong co du lieu
444,FUCTVGF4,True,0,NaN,Khong co du lieu
445,FUCTVGF5,True,0,NaN,Khong co du lieu
446,FUCVREIT,True,0,NaN,Khong co du lieu
...,...,...,...,...,...
2044,VFW,True,0,NaN,Khong co du lieu
2045,VHW,True,0,NaN,Khong co du lieu
2046,VLD,True,0,NaN,Khong co du lieu
2047,VTT,True,0,NaN,Khong co du lieu


In [6]:
log['CompanyType'].value_counts(dropna=False)

CompanyType
General       1848
NaN            116
Securities      42
Bank            31
Insurance       12
Name: count, dtype: int64

## Kiểm tra dữ liệu đã lưu

In [7]:
rows = []
with sqlite3.connect('fscore.db') as conn:
    tables = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' "
        "AND name LIKE 'fireant_financial_data_%' ORDER BY name")]
    for t in tables:
        n_sym, n_row, y0, y1 = conn.execute(
            f'SELECT COUNT(DISTINCT symbol), COUNT(*), MIN(Year), MAX(Year) '
            f'FROM "{t}"').fetchone()
        n_col = len(list(conn.execute(f'PRAGMA table_info("{t}")')))
        rows.append({'table': t, 'symbols': n_sym, 'rows': n_row,
                     'from': y0, 'to': y1, 'columns': n_col})

pd.DataFrame(rows)

,table,symbols,rows,from,to,columns
0,fireant_financial_data_bank,31,456,2009,2025,300
1,fireant_financial_data_general,1848,23671,2009,2025,342
2,fireant_financial_data_insurance,12,176,2009,2025,421
3,fireant_financial_data_securities,42,457,2014,2025,451


In [8]:
# đọc lại một mã (bảng rộng, giữ nguyên tên trường API)
hpg_db = Fireant.load_financial_data('fscore.db', symbol='HPG', company_type='General')
hpg_db[['symbol', 'period', 'NetSale', 'ProfitAfterTax', 'StockHolderEquity',
        'ROE', 'ROA', 'PE', 'BasicEPS', 'CashDividend', 'PiotroskiFScore']]

,symbol,period,NetSale,ProfitAfterTax,StockHolderEquity,ROE,ROA,PE,BasicEPS,CashDividend,PiotroskiFScore
0,HPG,2009,8123394614746,1270706623417,5064948541660,0.27149,0.16016,9.03169,6477.19354,2000,4
1,HPG,2010,14267083816361,1376316086778,6737989072718,0.22864,0.10731,9.11633,5103.98977,2000,3
2,HPG,2011,17851896561575,1296850503678,7963631176519,0.16820,0.07626,4.43882,3917.58057,0,4
3,HPG,2012,16826851892984,1030505429517,8577557545150,0.12019,0.05441,8.85301,2780.62225,1000,4
4,HPG,2013,18934292150531,2010435402769,9586960019559,0.21517,0.09285,8.81341,4663.34739,1000,8
5,HPG,2014,25525348822713,3250214590204,11965339743609,0.29178,0.13923,8.12312,6799.13785,1500,7
6,HPG,2015,27452932114333,3504382487779,14466710385310,0.26373,0.14646,6.13982,5383.28030,1000,7
7,HPG,2016,33283210159987,6606202726929,19850261077964,0.38477,0.22482,5.50815,8556.36677,1500,7
8,HPG,2017,46161691614304,8014756586048,32397580211910,0.30649,0.18566,8.87600,6157.22569,0,4
9,HPG,2018,55836458379759,8600550706227,40622949840810,0.23481,0.13064,7.66766,4622.81549,0,2


In [9]:
# gộp mọi loại hình: các trường không dùng cho loại hình đó sẽ là NaN
allfd = Fireant.load_financial_data('fscore.db')
print(allfd.shape, '|', allfd['symbol'].nunique(), 'mã')
allfd.groupby('CompanyType')['symbol'].nunique()

(24760, 873) | 1933 mã


CompanyType
Bank            31
General       1848
Insurance       12
Securities      42
Name: symbol, dtype: int64

In [10]:
# độ phủ theo năm của từng loại hình
allfd.pivot_table(index='Year', columns='CompanyType', values='symbol',
                  aggfunc='nunique')

CompanyType,Bank,General,Insurance,Securities
Year,,,,
2009,11.0,924.0,8.0,NaN
2010,22.0,981.0,8.0,NaN
2011,26.0,1054.0,8.0,NaN
2012,26.0,1110.0,8.0,NaN
2013,26.0,1203.0,8.0,NaN
2014,27.0,1331.0,8.0,24.0
2015,28.0,1464.0,10.0,27.0
2016,29.0,1577.0,11.0,37.0
2017,29.0,1609.0,12.0,37.0


## Cập nhật / lấy thêm theo quý

* Cập nhật số liệu năm mới: chạy lại với `skip_existing=False` — `INSERT OR REPLACE` theo
  `(symbol, period)` nên ghi đè chứ không nhân dòng.
* Dữ liệu quý ghi vào **cùng bảng** với `period` dạng `'Q1-2026'`, không đụng khoá của bản năm.

In [11]:
# # cập nhật lại toàn bộ (ghi đè các kỳ đã có)
# log_update = fa.crawl_financial_data_to_db(
#     symbols, start_year=2009, db_path='fscore.db', skip_existing=False,
#     log_path='./logs/fireant_financial_data_update.csv', sleep=0.3)
# log_update

In [12]:
# # bản theo quý
# log_q = fa.crawl_financial_data_to_db(
#     symbols, start_year=2009, type_time='QUY', db_path='fscore.db',
#     skip_existing=False, log_path='./logs/fireant_financial_data_quy.csv', sleep=0.3)
# log_q